# Quantalytics — Automated Quarantine QA (safe, isolated)
This runs the **same IBM Silver transformation** on two fake Bronze records without writing to any production Bronze or Silver table. It writes only to QA-labelled Delta tables (`workspace.silver.qa_*`). Run the cells from top to bottom. No credentials required.

In [0]:
# QA only: isolate all output tables. Production Bronze and Silver remain untouched.
import json
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, MapType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")
DATASET = "ibm"
MODE = "full"
SOURCE_FILE = ""
START_DATE = ""
END_DATE = ""
RUN_ID = str(uuid.uuid4())

# Real Bronze is READ only; all writes use distinctly named QA tables.
PRODUCTION_BRONZE = "workspace.bronze.ibm_calibration_raw"
IBM_BRONZE = "qa_ibm_invalid_bronze"   # session-local temporary view
IBM_SILVER = "workspace.silver.qa_ibm_silver_only"
BAD_TABLE = "workspace.silver.qa_quarantined_records_only"
LOG_TABLE = "workspace.silver.qa_pipeline_logs_only"

assert spark.catalog.tableExists(PRODUCTION_BRONZE), "First make sure IBM Bronze exists."
print("Using Bronze as read-only sample source:", PRODUCTION_BRONZE)


Using Bronze as read-only sample source: workspace.bronze.ibm_calibration_raw


In [0]:
# Create two fake Bronze rows, based on one authentic record's structure.
# No INSERT, UPDATE, or DELETE is issued against your real Bronze data.
base = spark.table(PRODUCTION_BRONZE).limit(1).first().asDict(recursive=True)
payload = json.loads(base["raw_payload"])

bad_type = dict(base)
bad_type_payload = dict(payload)
bad_type_payload["T1 (us)"] = "not_a_number"
bad_type["raw_payload"] = json.dumps(bad_type_payload)
bad_type["source_file"] = "QA_AUTO_BAD_NUMBER.csv"

new_column = dict(base)
new_column_payload = dict(payload)
new_column_payload["qa_unexpected_column"] = "unexpected"
new_column["raw_payload"] = json.dumps(new_column_payload)
new_column["source_file"] = "QA_AUTO_NEW_COLUMN.csv"

qa_bronze = spark.createDataFrame(
    [bad_type, new_column], schema=spark.table(PRODUCTION_BRONZE).schema
)
qa_bronze.createOrReplaceTempView(IBM_BRONZE)
print("Fake Bronze input rows:", spark.table(IBM_BRONZE).count())

# Explicit schemas for isolated QA output tables (same as the real notebook).
LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("stage", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("mode", StringType(), False),
    StructField("parameters", StringType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("input_rows", LongType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_message", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])
BAD_SCHEMA = StructType([
    StructField("record_key", StringType(), True),
    StructField("dataset", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("error_reason", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])
spark.createDataFrame([], LOG_SCHEMA).write.format("delta").mode("ignore").saveAsTable(LOG_TABLE)
spark.createDataFrame([], BAD_SCHEMA).write.format("delta").mode("ignore").saveAsTable(BAD_TABLE)
print("Isolated QA tables ready.")


Fake Bronze input rows: 2
Isolated QA tables ready.


In [0]:
# These are THE SAME functions used by the real Silver notebook.
def select_input(table, time_col):
    """Read Bronze, then optionally select one file or historical date range."""
    df = spark.table(table)
    if SOURCE_FILE:
        df = df.filter(F.col("source_file") == SOURCE_FILE)
    event_time = F.expr(f"try_cast({time_col} AS timestamp)")
    if START_DATE:
        df = df.filter(F.to_date(event_time) >= F.lit(START_DATE))
    if END_DATE:
        df = df.filter(F.to_date(event_time) <= F.lit(END_DATE))
    return df


def quarantine(bad_rows, dataset):
    """Keep faulty records for inspection; MERGE avoids duplicate bad rows."""
    rejected = (bad_rows.select("source_file", "raw_payload", "error_reason")
        .withColumn("dataset", F.lit(dataset))
        .withColumn("record_key", F.sha2(F.concat_ws("||", "dataset", "source_file", "raw_payload", "error_reason"), 256))
        .withColumn("load_timestamp", F.current_timestamp())
        .select("record_key", "dataset", "source_file", "raw_payload", "error_reason", "load_timestamp")
        .dropDuplicates(["record_key"]))
    n = rejected.count()
    if n:
        (DeltaTable.forName(spark, BAD_TABLE).alias("t")
         .merge(rejected.alias("s"), "t.record_key = s.record_key")
         .whenNotMatchedInsertAll()
         .execute())
    return n


def merge_silver(rows, table):
    """Create target if needed; update ONLY if actual source payload changed."""
    if not spark.catalog.tableExists(table):
        rows.limit(0).write.format("delta").saveAsTable(table)
    changes = {column: f"s.`{column}`" for column in rows.columns}
    (DeltaTable.forName(spark, table).alias("t")
     .merge(rows.alias("s"), "t.record_key = s.record_key")
     .whenMatchedUpdate(condition="t.source_payload_hash <> s.source_payload_hash", set=changes)
     .whenNotMatchedInsertAll()
     .execute())
    # Databricks reports how many rows the last MERGE inserted / updated.
    history = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()
    metrics = (history["operationMetrics"] or {}) if history else {}
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def audit(dataset, started, status, total=0, inserted=0, updated=0, bad=0, error=None):
    """Every attempted Bronze-to-Silver table run receives one log row."""
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    parameters = f"source_file={SOURCE_FILE or 'ALL'}; start_date={START_DATE or 'ALL'}; end_date={END_DATE or 'ALL'}"
    row = [(RUN_ID, "Bronze-to-Silver", dataset, MODE, parameters, started, now,
            status, int(total), int(inserted), int(updated), int(bad), error, now)]
    spark.createDataFrame(row, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)

In [0]:
# These are THE SAME IBM checks and transformation logic used by real Cell 4.
# These EXACT names come from the IBM CSV your friend used for Bronze.
IBM_COLUMNS = [
    "Qubit", "T1 (us)", "T2 (us)", "Readout assignment error", "Init error",
    "Prob meas0 prep1", "Prob meas1 prep0", "Readout length (ns)",
    "ID error", "Single-qubit gate length (ns)", "RX error",
    "Z-axis rotation (rz) error", "√x (sx) error", "Pauli-X error",
    "XSLOW error", "CZ error", "Gate length (ns)", "RZZ error",
    "MEASURE error", "MEASURE_2 error", "Operational", "Backend_ID",
    "Calibration_Timestamp"
]
IBM_PAYLOAD_SCHEMA = StructType([StructField(c, StringType(), True) for c in IBM_COLUMNS])

if DATASET in ("both", "ibm"):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    total = bad_n = 0
    try:
        bronze = select_input(IBM_BRONZE, "raw_calibration_timestamp")
        total = bronze.count()

        # from_json imposes the explicit StructType. A MapType separately spots extra fields.
        df = (bronze
          .withColumn("p", F.from_json("raw_payload", IBM_PAYLOAD_SCHEMA))
          .withColumn("all_keys", F.map_keys(F.from_json("raw_payload", MapType(StringType(), StringType()))))
          .withColumn("extra_keys", F.array_except("all_keys", F.array(*[F.lit(c) for c in IBM_COLUMNS])))
          .select(
            "raw_payload", "source_file", "collection_timestamp", "extra_keys",
            F.trim(F.col("p.Backend_ID")).alias("backend_id"),
            F.col("p.Qubit").alias("qubit_text"),
            F.col("p.Calibration_Timestamp").alias("time_text"),
            F.col("p.`T1 (us)`").alias("t1_text"),
            F.col("p.`T2 (us)`").alias("t2_text"),
            F.col("p.`Readout assignment error`").alias("readout_text"),
            F.col("p.`Init error`").alias("init_text"),
            F.col("p.Operational").alias("operational_text")))

        df = (df
          .withColumn("qubit_id", F.expr("try_cast(qubit_text AS INT)"))
          .withColumn("calibration_timestamp", F.expr("try_cast(time_text AS TIMESTAMP)"))
          .withColumn("t1_us", F.expr("try_cast(t1_text AS DOUBLE)"))
          .withColumn("t2_us", F.expr("try_cast(t2_text AS DOUBLE)"))
          .withColumn("readout_error", F.expr("try_cast(readout_text AS DOUBLE)"))
          .withColumn("init_error", F.expr("try_cast(init_text AS DOUBLE)"))
          .withColumn("operational", F.when(F.lower(F.trim("operational_text")).isin("yes", "y", "true", "1"), True)
                                       .when(F.lower(F.trim("operational_text")).isin("no", "n", "false", "0"), False)))

        # A null source measurement is allowed, but a non-empty invalid number is not.
        problems = [
            F.when(F.col("backend_id").isNull() | (F.col("backend_id") == ""), "MISSING_BACKEND"),
            F.when(F.col("qubit_id").isNull() | (F.col("qubit_id") < 0), "INVALID_QUBIT"),
            F.when(F.col("calibration_timestamp").isNull(), "INVALID_TIMESTAMP"),
            F.when(F.size("extra_keys") > 0, "SCHEMA_DRIFT_NEW_COLUMN"),
            F.when(F.col("raw_payload").isNull(), "NULL_PAYLOAD"),
        ]
        for text_col, numeric_col in [("t1_text", "t1_us"), ("t2_text", "t2_us"),
                                      ("readout_text", "readout_error"), ("init_text", "init_error")]:
            problems.append(F.when((F.trim(F.col(text_col)) != "") & F.col(text_col).isNotNull()
                                   & F.col(numeric_col).isNull(), f"INVALID_{numeric_col.upper()}"))
        problems += [
            F.when(F.col("t1_us").isNotNull() & (F.col("t1_us") <= 0), "T1_NOT_POSITIVE"),
            F.when(F.col("t2_us").isNotNull() & (F.col("t2_us") <= 0), "T2_NOT_POSITIVE"),
            F.when(F.col("readout_error").isNotNull() & ~F.col("readout_error").between(0, 1), "READOUT_OUT_OF_RANGE"),
            F.when(F.col("init_error").isNotNull() & ~F.col("init_error").between(0, 1), "INIT_OUT_OF_RANGE"),
            F.when(F.col("operational_text").isNotNull() & (F.trim("operational_text") != "")
                   & F.col("operational").isNull(), "INVALID_OPERATIONAL"),
        ]
        df = df.withColumn("error_reason", F.concat_ws("; ", *problems))
        bad_n = quarantine(df.filter(F.col("error_reason") != ""), "ibm")
        good = df.filter(F.col("error_reason") == "")

        # Key = one qubit, in one processor, at one calibration timestamp.
        ready = (good
          .withColumn("record_key", F.sha2(F.concat_ws("||", "backend_id",
                           F.col("calibration_timestamp").cast("string"),
                           F.col("qubit_id").cast("string")), 256))
          .withColumn("source_payload_hash", F.sha2("raw_payload", 256))
          .withColumn("load_timestamp", F.current_timestamp())
          .select("record_key", "backend_id", "calibration_timestamp", "qubit_id",
                  "t1_us", "t2_us", "readout_error", "init_error", "operational",
                  "source_payload_hash", "source_file", "load_timestamp"))
        # A source file might have repeated keys: take one candidate per key.
        w = Window.partitionBy("record_key").orderBy(F.col("source_file").desc(),
                                                      F.col("source_payload_hash").desc())
        ready = ready.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
        inserted, updated = merge_silver(ready, IBM_SILVER)
        audit("ibm", started, "SUCCESS", total, inserted, updated, bad_n)
        print(f"IBM SUCCESS: read {total}, inserted {inserted}, updated {updated}, quarantined {bad_n}")
        display(spark.table(IBM_SILVER).limit(10))
    except Exception as e:
        audit("ibm", started, "FAILURE", total, bad=bad_n, error=str(e)[:1500])
        raise

IBM SUCCESS: read 2, inserted 0, updated 0, quarantined 2


record_key,backend_id,calibration_timestamp,qubit_id,t1_us,t2_us,readout_error,init_error,operational,source_payload_hash,source_file,load_timestamp


In [0]:
# Verify the real pipeline logic automatically detected and quarantined both records.
qa_bad = spark.table(BAD_TABLE).filter(F.col("source_file").startswith("QA_AUTO_"))
display(qa_bad.select("dataset", "source_file", "error_reason", "load_timestamp"))

assert qa_bad.filter(F.col("source_file") == "QA_AUTO_BAD_NUMBER.csv").filter(
    F.col("error_reason").contains("INVALID_T1_US")).count() == 1, "Bad number not quarantined"
assert qa_bad.filter(F.col("source_file") == "QA_AUTO_NEW_COLUMN.csv").filter(
    F.col("error_reason").contains("SCHEMA_DRIFT_NEW_COLUMN")).count() == 1, "New column not quarantined"
assert spark.table(IBM_SILVER).count() == 0, "Invalid rows should not reach QA Silver"

print("PASS: 2 bad records were automatically quarantined.")
print("PASS: 0 bad rows landed in QA Silver.")
print("Your real Bronze and Silver tables were not modified by this test.")


dataset,source_file,error_reason,load_timestamp
ibm,QA_AUTO_BAD_NUMBER.csv,INVALID_T1_US,2026-10-10T11:22:46.823Z
ibm,QA_AUTO_NEW_COLUMN.csv,SCHEMA_DRIFT_NEW_COLUMN,2026-10-10T11:22:46.823Z


PASS: 2 bad records were automatically quarantined.
PASS: 0 bad rows landed in QA Silver.
Your real Bronze and Silver tables were not modified by this test.
